# UPLIFT MODEL SCRIPT (T-LEARNER WITH XGBOOST)
Reads from Excel `.xlsx`  
Outcome: `outcome_ed_90d`  
Treatment: `intervention_flag`


---
## 1. Install packages if needed
---


In [1]:
from pathlib import Path
import importlib.util
import sys

required_imports = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'xgboost': 'xgboost',
    'openpyxl': 'openpyxl',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
}
missing = [pip_name for import_name, pip_name in required_imports.items() if importlib.util.find_spec(import_name) is None]
if missing:
    raise ImportError('Install missing packages with: pip install ' + ' '.join(missing))

for candidate in [Path.cwd(), Path.cwd() / 'Code']:
    if (candidate / '_prism_model_utils.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not find _prism_model_utils.py in the notebook folder or ./Code')


---
## 2. Load packages
---


In [2]:
from itertools import product
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from _prism_model_utils import (
    GITHUB_XLSX_URL,
    align_to_columns,
    assert_xgb_booster_uses_cuda,
    clean_names_simple,
    ensure_output_folder,
    impute_categorical,
    impute_numeric,
    make_design_matrix,
    ntile_desc,
    project_root,
    read_prism_excel,
    require_columns,
    resolve_xgb_gpu_params,
    shap_importance_frame,
    split_train_test,
    to_binary,
    xgb_importance_frame,
    xgb_training_params,
)

warnings.filterwarnings('ignore', category=ConvergenceWarning)
PROJECT_ROOT = project_root()


---
## 3. FILE PATHS
---


In [3]:
# Raw GitHub URL to the Excel file
github_xlsx_url = GITHUB_XLSX_URL

# Output paths: keep XGBoost and GLMNET artifacts separate.
output_folder = ensure_output_folder(PROJECT_ROOT / 'Outputs' / 'Uplift' / 'Python')
xgboost_output_folder = ensure_output_folder(output_folder / 'XGBoost')
glmnet_output_folder = ensure_output_folder(output_folder / 'GLMNet')

xgboost_output_path = xgboost_output_folder / 'uplift_scored_output.csv'
xgboost_summary_path = xgboost_output_folder / 'uplift_decile_summary.csv'
glmnet_output_path = glmnet_output_folder / 'uplift_scored_output.csv'
glmnet_summary_path = glmnet_output_folder / 'uplift_decile_summary.csv'

# Backward-compatible aliases for the primary XGBoost output.
output_path = xgboost_output_path
summary_path = xgboost_summary_path

print('Imported data from:', github_xlsx_url, '\n')
print('Project root resolved to:', PROJECT_ROOT, '\n')
print('Python output root:', output_folder, '\n')
print('XGBoost outputs will be saved to:', xgboost_output_folder, '\n')
print('GLMNET outputs will be saved to:', glmnet_output_folder, '\n')


---
## 4. HELPER FUNCTIONS
---


In [4]:
def safe_as_date(values):
    return pd.to_datetime(values, errors='coerce')


def present_columns(columns, df):
    return [column for column in columns if column in df.columns]


def safe_auc(y_true, y_pred, label):
    y_series = pd.Series(y_true).dropna()
    if y_series.nunique() < 2:
        print(f'{label} AUC: cannot calculate because only one outcome class is present')
        return np.nan
    auc_value = roc_auc_score(y_true, y_pred)
    print(f'{label} AUC: {auc_value:.4f}')
    return auc_value


REQUIRE_GPU_FOR_XGBOOST = True
RUN_CPU_ONLY_COMPARISON_MODELS = True
XGBOOST_CUDA_DEVICE = 0
XGB_GPU_PARAMS = resolve_xgb_gpu_params(cuda_device=XGBOOST_CUDA_DEVICE) if REQUIRE_GPU_FOR_XGBOOST else {}
print('XGBoost GPU required:', REQUIRE_GPU_FOR_XGBOOST)
print('XGBoost CUDA params used for training:', XGB_GPU_PARAMS)
print('CPU-only GLMNET comparison enabled:', RUN_CPU_ONLY_COMPARISON_MODELS)


def make_dmatrix(x_matrix, y=None):
    if y is None:
        return xgb.DMatrix(x_matrix, feature_names=list(x_matrix.columns))
    return xgb.DMatrix(x_matrix, label=np.asarray(y, dtype=float), feature_names=list(x_matrix.columns))


def fit_xgb_cv_grid(x_matrix, y, grid, nrounds_max=500, nfold=5, seed=123):
    y_array = np.asarray(y, dtype=float)
    dtrain = make_dmatrix(x_matrix, y_array)
    class_counts = pd.Series(y_array).value_counts()
    folds = int(min(nfold, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two outcome classes with at least two rows each for XGBoost CV.')

    results = []
    best_model_info = None
    best_auc = -np.inf

    for params_grid in grid:
        params_i = xgb_training_params(
            XGB_GPU_PARAMS,
            {
                'max_depth': params_grid['max_depth'],
                'eta': params_grid['eta'],
                'min_child_weight': params_grid['min_child_weight'],
                'subsample': 0.8,
                'colsample_bytree': 0.8,
            },
            eval_metric='auc',
            seed=seed,
        )
        cv_i = xgb.cv(
            params=params_i,
            dtrain=dtrain,
            num_boost_round=nrounds_max,
            nfold=folds,
            stratified=True,
            early_stopping_rounds=20,
            seed=seed,
            verbose_eval=False,
        )
        auc_column = 'test-auc-mean'
        best_iter_i = int(cv_i[auc_column].idxmax())
        best_auc_i = float(cv_i.loc[best_iter_i, auc_column])
        best_nrounds_i = best_iter_i + 1
        results.append({
            'max_depth': params_grid['max_depth'],
            'eta': params_grid['eta'],
            'min_child_weight': params_grid['min_child_weight'],
            'best_nrounds': best_nrounds_i,
            'cv_auc': best_auc_i,
        })
        if best_auc_i > best_auc:
            best_auc = best_auc_i
            best_model_info = {'params': params_i, 'best_nrounds': best_nrounds_i, 'cv_auc': best_auc_i}

    search_results = pd.DataFrame(results).sort_values('cv_auc', ascending=False).reset_index(drop=True)
    final_model = xgb.train(
        params=best_model_info['params'],
        dtrain=dtrain,
        num_boost_round=best_model_info['best_nrounds'],
        verbose_eval=False,
    )
    return {
        'model': final_model,
        'best_params': best_model_info['params'],
        'best_nrounds': best_model_info['best_nrounds'],
        'best_cv_auc': best_model_info['cv_auc'],
        'search_results': search_results,
    }


def fit_elastic_net(x_matrix, y, alpha_grid=np.round(np.arange(0, 1.01, 0.1), 1), nfolds=5, seed=123):
    y_array = np.asarray(y, dtype=float)
    class_counts = pd.Series(y_array).value_counts()
    folds = int(min(nfolds, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two outcome classes with at least two rows each for elastic-net CV.')

    if not RUN_CPU_ONLY_COMPARISON_MODELS:
        raise RuntimeError(
            'fit_elastic_net uses sklearn LogisticRegressionCV, which trains on CPU. '
            'Set RUN_CPU_ONLY_COMPARISON_MODELS = True to run this CPU comparison model.'
        )

    results = []
    best_pipeline = None
    best_auc = -np.inf
    best_alpha = np.nan
    best_lambda = np.nan
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)

    for alpha in alpha_grid:
        penalty = 'l2' if alpha == 0 else 'elasticnet'
        l1_ratios = None if alpha == 0 else [float(alpha)]
        cv_model = LogisticRegressionCV(
            Cs=np.logspace(-4, 4, 30),
            cv=cv,
            penalty=penalty,
            solver='saga',
            l1_ratios=l1_ratios,
            scoring='roc_auc',
            max_iter=10000,
            random_state=seed,
            refit=True,
        )
        pipeline = make_pipeline(StandardScaler(), cv_model)
        pipeline.fit(x_matrix, y_array)
        fitted = pipeline.named_steps['logisticregressioncv']
        scores = fitted.scores_[1.0]
        auc_cv = float(np.nanmax(np.nanmean(scores, axis=0)))
        lambda_value = float(1 / fitted.C_[0])
        results.append({'alpha': float(alpha), 'lambda': lambda_value, 'cv_auc': auc_cv})
        if auc_cv > best_auc:
            best_auc = auc_cv
            best_alpha = float(alpha)
            best_lambda = lambda_value
            best_pipeline = pipeline

    return {
        'best_model': best_pipeline,
        'best_alpha': best_alpha,
        'best_lambda': best_lambda,
        'best_auc': best_auc,
        'search_results': pd.DataFrame(results).sort_values('cv_auc', ascending=False).reset_index(drop=True),
    }


def build_uplift_results(base_df, pred_treated, pred_control):
    results = base_df.copy()
    results['pred_ed_if_treated'] = pred_treated
    results['pred_ed_if_control'] = pred_control
    results['benefit_score'] = results['pred_ed_if_control'] - results['pred_ed_if_treated']
    results['uplift_bad_outcome'] = results['pred_ed_if_treated'] - results['pred_ed_if_control']
    results['uplift_decile'] = ntile_desc(results['benefit_score'], 10).to_numpy()
    return results


def summarize_uplift_deciles(results):
    return (
        results
        .groupby('uplift_decile', as_index=False)
        .agg(
            n=('outcome_ed_90d', 'size'),
            avg_benefit_score=('benefit_score', 'mean'),
            observed_ed_rate=('outcome_ed_90d', 'mean'),
            treated_pct=('intervention_flag', 'mean'),
            avg_pred_ed_if_treated=('pred_ed_if_treated', 'mean'),
            avg_pred_ed_if_control=('pred_ed_if_control', 'mean'),
        )
        .sort_values('uplift_decile')
    )


def print_highest_benefit(results, label, n=20):
    print(f'{label} top {n} highest-benefit members:')
    display(
        results
        .sort_values('benefit_score', ascending=False)
        [[
            'outcome_ed_90d',
            'intervention_flag',
            'pred_ed_if_treated',
            'pred_ed_if_control',
            'benefit_score',
            'uplift_decile',
        ]]
        .head(n)
    )
    print()


def glmnet_contribution_importance_frame(model_info, x_matrix, label):
    pipeline = model_info['best_model']
    scaler = pipeline.named_steps['standardscaler']
    glmnet_model = pipeline.named_steps['logisticregressioncv']
    x_scaled = scaler.transform(x_matrix)
    coefficients = glmnet_model.coef_.ravel()
    contributions = x_scaled * coefficients
    return (
        pd.DataFrame(
            {
                'feature': list(x_matrix.columns),
                'mean_abs_model_contribution': np.abs(contributions).mean(axis=0),
                'coefficient': coefficients,
                'model': label,
                'importance_type': 'standardized_logit_contribution',
            }
        )
        .sort_values('mean_abs_model_contribution', ascending=False)
        .reset_index(drop=True)
    )



---
## 5. READ EXCEL FILE
---


In [5]:
df_raw = read_prism_excel(github_xlsx_url)
df_raw.columns = clean_names_simple(df_raw.columns)

print('Rows:', len(df_raw))
print('Columns:', len(df_raw.columns))
print()
print('Column names after cleaning:')
print(list(df_raw.columns))
print()


---
## 6. CHECK REQUIRED COLUMNS
---


In [6]:
required_fields = ['outcome_ed_90d', 'intervention_flag']
require_columns(df_raw, required_fields)


---
## 7. BASIC CLEANUP
---


In [7]:
df = df_raw.copy()

for column in ['index_date', 'intervention_start_date', 'intervention_end_date']:
    if column in df.columns:
        df[column] = safe_as_date(df[column])

df['intervention_flag'] = to_binary(df['intervention_flag'])
df['outcome_ed_90d'] = to_binary(df['outcome_ed_90d'])


---
## 8. DERIVE DATE FEATURES
---


In [8]:
if 'intervention_start_date' in df.columns:
    df['intervention_start_month'] = df['intervention_start_date'].dt.month.astype(float)
    df['intervention_start_wday'] = (((df['intervention_start_date'].dt.dayofweek + 1) % 7) + 1).astype(float)
else:
    df['intervention_start_month'] = np.nan
    df['intervention_start_wday'] = np.nan

if {'index_date', 'intervention_start_date'}.issubset(df.columns):
    df['days_to_intervention_start'] = (df['intervention_start_date'] - df['index_date']).dt.days.astype(float)
else:
    df['days_to_intervention_start'] = np.nan

if {'intervention_start_date', 'intervention_end_date'}.issubset(df.columns):
    df['intervention_duration_calc'] = (df['intervention_end_date'] - df['intervention_start_date']).dt.days.astype(float)
else:
    df['intervention_duration_calc'] = np.nan

if 'intervention_days_active' not in df.columns:
    df['intervention_days_active'] = df['intervention_duration_calc']

# reverse for now
df = df_raw.copy()


---
## 9. SELECT PREDICTORS
---


In [9]:
candidate_predictors_all = [
    'client_contract', 'service_region', 'program', 'case_manager_name', 'age', 'gender',
    'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag', 'diabetes_flag',
    'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag', 'anxiety_flag',
    'substance_use_flag', 'ckd_flag', 'pregnancy_flag', 'behavioral_health_risk_flag',
    'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag',
    'utilities_insecurity_flag', 'pcp_visits_last_6m', 'specialist_visits_last_6m',
    'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m',
    'total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag',
    'opioid_flag', 'polypharmacy_flag', 'percolator_utilization_score',
    'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score', 'risk_tier',
    'intervention_type', 'intervention_days_active', 'touches_per_month', 'outreach_attempts',
    'successful_contacts', 'avg_call_duration_min', 'max_call_duration_min', 'notes_escalation_flag',
    'community_referral_flag', 'pharmacy_review_flag', 'engagement_level',
    'days_to_intervention_start', 'intervention_start_month', 'intervention_start_wday',
]

candidate_predictors = [column for column in candidate_predictors_all if column in df.columns]
missing_predictors = [column for column in candidate_predictors_all if column not in df.columns]

if missing_predictors:
    print('Predictors not found in dataset:')
    print(missing_predictors)
else:
    print('All candidate predictors are present in dataset.')

model_df = df[['outcome_ed_90d', 'intervention_flag', *candidate_predictors]].copy()
model_df['outcome_ed_90d'] = to_binary(model_df['outcome_ed_90d'])
model_df['intervention_flag'] = to_binary(model_df['intervention_flag'])
model_df = model_df[model_df['outcome_ed_90d'].notna() & model_df['intervention_flag'].notna()].copy()

missing_in_model_df = [column for column in df.columns if column not in model_df.columns]
if missing_in_model_df:
    print('Columns in dataframe but not in model:')
    print(missing_in_model_df)
else:
    print('All dataframe columns are present in model.')
print('Modeling rows after dropping missing outcome/treatment:', len(model_df))
print()


---
## 10. DATA TYPE HANDLING
---


In [10]:
model_df.columns

In [11]:
flag_like_cols = [column for column in model_df.columns if column.endswith('_flag')]
for column in flag_like_cols:
    model_df[column] = to_binary(model_df[column])

possible_numeric_cols = [
    'age', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d',
    'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m',
    'rx_count_last_6m', 'med_adherence_pdc', 'percolator_utilization_score',
    'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score',
    'intervention_days_active', 'touches_per_month', 'outreach_attempts', 'successful_contacts',
    'avg_call_duration_min', 'max_call_duration_min', 'days_to_intervention_start',
    'intervention_start_month', 'intervention_start_wday',
]

for column in present_columns(possible_numeric_cols, model_df):
    model_df[column] = pd.to_numeric(model_df[column], errors='coerce')

for column in model_df.columns:
    if column in ['outcome_ed_90d', 'intervention_flag']:
        continue
    if not pd.api.types.is_numeric_dtype(model_df[column]):
        model_df[column] = impute_categorical(model_df[column])

for column in model_df.columns:
    if column in ['outcome_ed_90d', 'intervention_flag']:
        continue
    if pd.api.types.is_numeric_dtype(model_df[column]):
        model_df[column] = impute_numeric(model_df[column])

unique_counts = model_df.apply(lambda column: column.dropna().nunique())
keep_cols = list(unique_counts[unique_counts > 1].index)
model_df = model_df.loc[:, keep_cols].copy().reset_index(drop=True)

print('Final modeling columns:')
print(list(model_df.columns))
print()


---
## 11. TRAIN / TEST SPLIT
---


In [12]:
train_df, test_df = split_train_test(
    model_df,
    train_fraction=0.70,
    seed=123,
    stratify_columns=['intervention_flag', 'outcome_ed_90d'],
)

print('Training rows:', len(train_df))
print('Testing rows:', len(test_df))
print()


---
## 12. SEPARATE TREATED / CONTROL
---


In [13]:
train_treated = train_df[train_df['intervention_flag'] == 1].copy()
train_control = train_df[train_df['intervention_flag'] == 0].copy()

print('Training treated rows:', len(train_treated))
print('Training control rows:', len(train_control))
print()

if len(train_treated) < 50:
    raise ValueError('Too few treated rows to train a stable model.')
if len(train_control) < 50:
    raise ValueError('Too few control rows to train a stable model.')


---
## 13. BUILD MODEL MATRICES
---


In [14]:
feature_cols = [column for column in model_df.columns if column not in ['outcome_ed_90d', 'intervention_flag']]

train_treated_x_df = train_treated[feature_cols].copy()
train_control_x_df = train_control[feature_cols].copy()
test_x_df = test_df[feature_cols].copy()

combined_matrix, split_matrices = make_design_matrix([train_treated_x_df, train_control_x_df, test_x_df])
x_treated, x_control, x_test = split_matrices

y_treated = train_treated['outcome_ed_90d'].astype(float).to_numpy()
y_control = train_control['outcome_ed_90d'].astype(float).to_numpy()


---
## WRITE-UP DATA REVIEW SUMMARY
---


In [15]:
data_review_summary = pd.DataFrame(
    [
        {
            'total_members': len(model_df),
            'treated_members': int((model_df['intervention_flag'] == 1).sum()),
            'control_members': int((model_df['intervention_flag'] == 0).sum()),
            'treatment_rate': float(model_df['intervention_flag'].mean()),
            'outcome_events': int((model_df['outcome_ed_90d'] == 1).sum()),
            'outcome_prevalence': float(model_df['outcome_ed_90d'].mean()),
            'treated_outcome_rate': float(model_df.loc[model_df['intervention_flag'] == 1, 'outcome_ed_90d'].mean()),
            'control_outcome_rate': float(model_df.loc[model_df['intervention_flag'] == 0, 'outcome_ed_90d'].mean()),
            'number_of_predictors': len(feature_cols),
            'number_of_model_matrix_columns': x_test.shape[1],
        }
    ]
)

data_review_summary_path = output_folder / 'data_review_summary.csv'
data_review_summary.to_csv(data_review_summary_path, index=False)

print('Data review summary:')
display(data_review_summary)
print('Data review summary written to:', data_review_summary_path)


---
## WRITE-UP PREDICTOR DATA DICTIONARY AND SUMMARY
---


In [16]:
predictor_category_lookup = {
    # Demographics
    'client_contract': 'Demographics',
    'service_region': 'Demographics',
    'program': 'Demographics',
    'case_manager_name': 'Demographics',
    'age': 'Demographics',
    'gender': 'Demographics',
    'dual_eligible': 'Demographics',
    'county': 'Demographics',
    'plan_type': 'Demographics',
    'language': 'Demographics',
    'living_alone_flag': 'Demographics',

    # Clinical Conditions
    'diabetes_flag': 'Clinical Conditions',
    'chf_flag': 'Clinical Conditions',
    'copd_flag': 'Clinical Conditions',
    'asthma_flag': 'Clinical Conditions',
    'depression_flag': 'Clinical Conditions',
    'anxiety_flag': 'Clinical Conditions',
    'substance_use_flag': 'Clinical Conditions',
    'ckd_flag': 'Clinical Conditions',
    'pregnancy_flag': 'Clinical Conditions',
    'behavioral_health_risk_flag': 'Clinical Conditions',

    # SDOH
    'food_insecurity_flag': 'SDOH',
    'housing_instability_flag': 'SDOH',
    'transportation_barrier_flag': 'SDOH',
    'utilities_insecurity_flag': 'SDOH',

    # Utilization
    'pcp_visits_last_6m': 'Utilization',
    'specialist_visits_last_6m': 'Utilization',
    'ed_visits_last_30d': 'Utilization',
    'ed_visits_last_6m': 'Utilization',
    'admits_last_6m': 'Utilization',
    'observation_stays_last_6m': 'Utilization',

    # Pharmacy
    'total_cost_last_6m': 'Pharmacy',
    'rx_count_last_6m': 'Pharmacy',
    'med_adherence_pdc': 'Pharmacy',
    'high_cost_drug_flag': 'Pharmacy',
    'opioid_flag': 'Pharmacy',
    'polypharmacy_flag': 'Pharmacy',

    # Risk Scores
    'utilization_score': 'Risk Scores',
    'clinical_score': 'Risk Scores',
    'sdoh_score': 'Risk Scores',
    'percolator_utilization_score': 'Risk Scores',
    'percolator_clinical_score': 'Risk Scores',
    'percolator_sdoh_score': 'Risk Scores',
    'current_risk_score': 'Risk Scores',
    'risk_tier': 'Risk Scores',
}

predictor_description_lookup = {
    'client_contract': 'Client contract or line of business associated with the member.',
    'service_region': 'Service region where the member receives care management support.',
    'program': 'Care management program or operational program assignment.',
    'case_manager_name': 'Assigned case manager identifier/name.',
    'age': 'Member age at the index point.',
    'gender': 'Member gender category.',
    'dual_eligible': 'Indicator or category for Medicare/Medicaid dual eligibility.',
    'county': 'Member county of residence.',
    'plan_type': 'Health plan product or plan type.',
    'language': 'Preferred or primary language.',
    'living_alone_flag': 'Indicator that the member lives alone.',
    'diabetes_flag': 'Indicator for diabetes diagnosis/history.',
    'chf_flag': 'Indicator for congestive heart failure diagnosis/history.',
    'copd_flag': 'Indicator for COPD diagnosis/history.',
    'asthma_flag': 'Indicator for asthma diagnosis/history.',
    'depression_flag': 'Indicator for depression diagnosis/history.',
    'anxiety_flag': 'Indicator for anxiety diagnosis/history.',
    'substance_use_flag': 'Indicator for substance use-related risk or diagnosis.',
    'ckd_flag': 'Indicator for chronic kidney disease diagnosis/history.',
    'pregnancy_flag': 'Indicator for pregnancy status/history.',
    'behavioral_health_risk_flag': 'Indicator for behavioral health risk.',
    'food_insecurity_flag': 'Indicator for food insecurity.',
    'housing_instability_flag': 'Indicator for housing instability.',
    'transportation_barrier_flag': 'Indicator for transportation barriers.',
    'utilities_insecurity_flag': 'Indicator for utility insecurity.',
    'pcp_visits_last_6m': 'Number of primary care visits in the last 6 months.',
    'specialist_visits_last_6m': 'Number of specialist visits in the last 6 months.',
    'ed_visits_last_30d': 'Number of ED visits in the last 30 days.',
    'ed_visits_last_6m': 'Number of ED visits in the last 6 months.',
    'admits_last_6m': 'Number of inpatient admissions in the last 6 months.',
    'observation_stays_last_6m': 'Number of observation stays in the last 6 months.',
    'total_cost_last_6m': 'Total healthcare cost in the last 6 months.',
    'rx_count_last_6m': 'Number of prescriptions or medication fills in the last 6 months.',
    'med_adherence_pdc': 'Medication adherence proportion of days covered.',
    'high_cost_drug_flag': 'Indicator for high-cost drug use.',
    'opioid_flag': 'Indicator for opioid medication use.',
    'polypharmacy_flag': 'Indicator for polypharmacy risk.',
    'percolator_utilization_score': 'Composite utilization risk score.',
    'percolator_clinical_score': 'Composite clinical risk score.',
    'percolator_sdoh_score': 'Composite social determinants of health risk score.',
    'current_risk_score': 'Current overall risk score.',
    'risk_tier': 'Categorical risk tier assignment.',
    'intervention_type': 'Type of intervention assigned or delivered.',
    'intervention_days_active': 'Number of days the intervention was active.',
    'touches_per_month': 'Average number of care management touches per month.',
    'outreach_attempts': 'Number of outreach attempts.',
    'successful_contacts': 'Number of successful contacts.',
    'avg_call_duration_min': 'Average call duration in minutes.',
    'max_call_duration_min': 'Maximum call duration in minutes.',
    'notes_escalation_flag': 'Indicator for escalation noted in care management documentation.',
    'community_referral_flag': 'Indicator for community referral.',
    'pharmacy_review_flag': 'Indicator for pharmacy review.',
    'engagement_level': 'Member engagement level category.',
    'days_to_intervention_start': 'Days between index date and intervention start.',
    'intervention_start_month': 'Calendar month of intervention start.',
    'intervention_start_wday': 'Day of week of intervention start.',
}


def predictor_category(column):
    return predictor_category_lookup.get(column, 'Other / derived')


def predictor_description(column):
    return predictor_description_lookup.get(column, f'Model predictor derived from `{column}`.')


def compact_examples(series, max_items=5):
    values = series.dropna().astype(str).drop_duplicates().head(max_items).tolist()
    return '; '.join(values)


def top_value_summary(series, max_items=5):
    counts = series.dropna().astype(str).value_counts().head(max_items)
    return '; '.join(f'{idx} ({count})' for idx, count in counts.items())


predictor_dictionary_rows = []
for column in candidate_predictors:
    raw_series = df[column] if column in df.columns else pd.Series(dtype='object')
    model_series = model_df[column] if column in model_df.columns else raw_series
    numeric_values = pd.to_numeric(raw_series, errors='coerce')
    is_numeric_like = pd.api.types.is_numeric_dtype(model_series) or numeric_values.notna().sum() > 0
    model_matrix_matches = [matrix_col for matrix_col in x_test.columns if matrix_col == column or matrix_col.startswith(f'{column}_')]
    predictor_dictionary_rows.append(
        {
            'variable': column,
            'category': predictor_category(column),
            'data_type': str(raw_series.dtype),
            'model_ready_data_type': str(model_series.dtype),
            'description': predictor_description(column),
            'included_in_model': column in feature_cols,
            'included_in_model_matrix': len(model_matrix_matches) > 0,
            'model_matrix_columns': '; '.join(model_matrix_matches[:20]),
            'missing_count': int(raw_series.isna().sum()),
            'missing_pct': float(raw_series.isna().mean()),
            'unique_values': int(raw_series.dropna().nunique()),
            'example_values': compact_examples(raw_series),
            'min_value': float(numeric_values.min()) if is_numeric_like and numeric_values.notna().any() else np.nan,
            'max_value': float(numeric_values.max()) if is_numeric_like and numeric_values.notna().any() else np.nan,
        }
    )

predictor_data_dictionary = pd.DataFrame(predictor_dictionary_rows).sort_values(['category', 'variable']).reset_index(drop=True)
predictor_data_dictionary_path = output_folder / 'predictor_data_dictionary.csv'
predictor_data_dictionary.to_csv(predictor_data_dictionary_path, index=False)

numeric_summary_rows = []
categorical_summary_rows = []
for column in candidate_predictors:
    raw_series = df[column] if column in df.columns else pd.Series(dtype='object')
    numeric_values = pd.to_numeric(raw_series, errors='coerce')
    numeric_share = numeric_values.notna().mean() if len(raw_series) else 0
    treat_as_numeric = column in possible_numeric_cols or column.endswith('_flag') or numeric_share >= 0.90

    if treat_as_numeric and numeric_values.notna().any():
        numeric_summary_rows.append(
            {
                'variable': column,
                'category': predictor_category(column),
                'n': int(numeric_values.notna().sum()),
                'missing_count': int(raw_series.isna().sum()),
                'missing_pct': float(raw_series.isna().mean()),
                'mean': float(numeric_values.mean()),
                'median': float(numeric_values.median()),
                'std': float(numeric_values.std()),
                'min': float(numeric_values.min()),
                'p25': float(numeric_values.quantile(0.25)),
                'p75': float(numeric_values.quantile(0.75)),
                'max': float(numeric_values.max()),
            }
        )
    else:
        non_missing = raw_series.dropna().astype(str)
        value_counts = non_missing.value_counts()
        mode_value = value_counts.index[0] if len(value_counts) else np.nan
        mode_count = int(value_counts.iloc[0]) if len(value_counts) else 0
        categorical_summary_rows.append(
            {
                'variable': column,
                'category': predictor_category(column),
                'n': int(non_missing.shape[0]),
                'missing_count': int(raw_series.isna().sum()),
                'missing_pct': float(raw_series.isna().mean()),
                'unique_values': int(non_missing.nunique()),
                'mode': mode_value,
                'mode_count': mode_count,
                'mode_pct': float(mode_count / len(raw_series)) if len(raw_series) else np.nan,
                'top_values': top_value_summary(raw_series),
            }
        )

numeric_predictor_summary = pd.DataFrame(numeric_summary_rows).sort_values(['category', 'variable']).reset_index(drop=True)
categorical_predictor_summary = pd.DataFrame(categorical_summary_rows).sort_values(['category', 'variable']).reset_index(drop=True)

numeric_predictor_summary_path = output_folder / 'numeric_predictor_summary.csv'
categorical_predictor_summary_path = output_folder / 'categorical_predictor_summary.csv'
numeric_predictor_summary.to_csv(numeric_predictor_summary_path, index=False)
categorical_predictor_summary.to_csv(categorical_predictor_summary_path, index=False)

print('Predictor data dictionary:')
display(predictor_data_dictionary)
print('Predictor data dictionary written to:', predictor_data_dictionary_path)
print()

print('Numeric predictor summary:')
display(numeric_predictor_summary)
print('Numeric predictor summary written to:', numeric_predictor_summary_path)
print()

print('Categorical predictor summary:')
display(categorical_predictor_summary)
print('Categorical predictor summary written to:', categorical_predictor_summary_path)



---
## PREDICTOR DISTRIBUTION VISUALS
---


In [17]:
from matplotlib.backends.backend_pdf import PdfPages
import re

predictor_distribution_folder = ensure_output_folder(output_folder / 'Predictor_Distributions')
numeric_distribution_folder = ensure_output_folder(predictor_distribution_folder / 'Numeric')
categorical_distribution_folder = ensure_output_folder(predictor_distribution_folder / 'Categorical')

numeric_distribution_pdf_path = predictor_distribution_folder / 'numeric_predictor_distributions.pdf'
categorical_distribution_pdf_path = predictor_distribution_folder / 'categorical_predictor_distributions.pdf'
distribution_visual_index_path = predictor_distribution_folder / 'predictor_distribution_visual_index.csv'

sns.set_theme(style='whitegrid')


def safe_filename(value):
    safe_value = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')
    return safe_value or 'unnamed'


def plot_numeric_distribution(column, folder):
    values = pd.to_numeric(df[column], errors='coerce').dropna()
    if values.empty:
        return None

    fig, ax = plt.subplots(figsize=(8, 4.8))
    unique_count = values.nunique()
    if unique_count <= 15:
        sns.histplot(values, discrete=True, shrink=0.8, ax=ax, color='#4c78a8')
    else:
        sns.histplot(values, bins=30, kde=True, ax=ax, color='#4c78a8')
    ax.set_title(f'{column} Distribution')
    ax.set_xlabel(column)
    ax.set_ylabel('Member count')
    fig.tight_layout()

    output_path = folder / f'{safe_filename(column)}_histogram.png'
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    return fig, output_path


def plot_categorical_distribution(column, folder, top_n=20):
    values = df[column].dropna().astype(str)
    if values.empty:
        return None

    counts = values.value_counts()
    if len(counts) > top_n:
        counts = pd.concat([counts.head(top_n), pd.Series({'Other': counts.iloc[top_n:].sum()})])

    fig_height = max(4.8, min(12, 0.35 * len(counts) + 2.2))
    fig, ax = plt.subplots(figsize=(9, fig_height))
    plot_df = counts.reset_index()
    plot_df.columns = [column, 'member_count']
    sns.barplot(data=plot_df, x='member_count', y=column, ax=ax, color='#59a14f')
    ax.set_title(f'{column} Distribution')
    ax.set_xlabel('Member count')
    ax.set_ylabel(column)
    for patch in ax.patches:
        width = patch.get_width()
        if pd.notna(width):
            ax.annotate(
                f'{int(width):,}',
                (width, patch.get_y() + patch.get_height() / 2),
                ha='left',
                va='center',
                fontsize=8,
                xytext=(4, 0),
                textcoords='offset points',
            )
    fig.tight_layout()

    output_path = folder / f'{safe_filename(column)}_bar_chart.png'
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    return fig, output_path


distribution_visual_rows = []

numeric_columns_for_plots = numeric_predictor_summary['variable'].tolist() if not numeric_predictor_summary.empty else []
with PdfPages(numeric_distribution_pdf_path) as pdf:
    for column in numeric_columns_for_plots:
        result = plot_numeric_distribution(column, numeric_distribution_folder)
        if result is None:
            continue
        fig, output_path = result
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        distribution_visual_rows.append({
            'variable': column,
            'category': predictor_category(column),
            'visual_type': 'numeric_histogram',
            'output_file': str(output_path),
        })

categorical_columns_for_plots = categorical_predictor_summary['variable'].tolist() if not categorical_predictor_summary.empty else []
with PdfPages(categorical_distribution_pdf_path) as pdf:
    for column in categorical_columns_for_plots:
        result = plot_categorical_distribution(column, categorical_distribution_folder)
        if result is None:
            continue
        fig, output_path = result
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        distribution_visual_rows.append({
            'variable': column,
            'category': predictor_category(column),
            'visual_type': 'categorical_bar_chart',
            'output_file': str(output_path),
        })

distribution_visual_index = pd.DataFrame(distribution_visual_rows)
distribution_visual_index.to_csv(distribution_visual_index_path, index=False)

print('Predictor distribution visuals written to:', predictor_distribution_folder)
print('Numeric predictor distribution PDF:', numeric_distribution_pdf_path)
print('Categorical predictor distribution PDF:', categorical_distribution_pdf_path)
print('Distribution visual index written to:', distribution_visual_index_path)
display(distribution_visual_index)


---
## 14. TRAIN XGBOOST MODELS
---


In [18]:
print('Unique y_treated values:')
print(np.sort(pd.unique(y_treated)))
print()

print('Unique y_control values:')
print(np.sort(pd.unique(y_control)))
print()

if not set(pd.Series(y_treated).dropna().unique()).issubset({0.0, 1.0}):
    raise ValueError('y_treated contains values other than 0 and 1.')
if not set(pd.Series(y_control).dropna().unique()).issubset({0.0, 1.0}):
    raise ValueError('y_control contains values other than 0 and 1.')

dtrain_treated = make_dmatrix(x_treated, y_treated)
dtrain_control = make_dmatrix(x_control, y_control)

params = xgb_training_params(
    XGB_GPU_PARAMS,
    {
        'max_depth': 4,
        'eta': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    },
    eval_metric='logloss',
    seed=123,
)

model_treated = xgb.train(params=params, dtrain=dtrain_treated, num_boost_round=150, verbose_eval=False)
model_control = xgb.train(params=params, dtrain=dtrain_control, num_boost_round=150, verbose_eval=False)
assert_xgb_booster_uses_cuda(model_treated, 'Baseline treated XGBoost model')
assert_xgb_booster_uses_cuda(model_control, 'Baseline control XGBoost model')

print('Models trained successfully with XGBoost GPU params:', XGB_GPU_PARAMS)
print()

test_treated_pos = np.where(test_df['intervention_flag'].to_numpy() == 1)[0]
test_control_pos = np.where(test_df['intervention_flag'].to_numpy() == 0)[0]

pred_treated = model_treated.predict(make_dmatrix(x_test.iloc[test_treated_pos]))
auc_treated = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated, 'XGBoost Treated model')

pred_control = model_control.predict(make_dmatrix(x_test.iloc[test_control_pos]))
auc_control = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control, 'XGBoost Control model')


---
## XGBOOST CV GRID SEARCH FUNCTION
---


In [19]:
xgb_grid = [
    {'max_depth': max_depth, 'eta': eta, 'min_child_weight': min_child_weight}
    for max_depth, eta, min_child_weight in product([3, 4, 5], [0.03, 0.05, 0.10], [1, 5])
]


---
## TRAIN TREATED MODEL WITH CV GRID SEARCH
---


In [20]:
xgb_treated_cv = fit_xgb_cv_grid(x_matrix=x_treated, y=y_treated, grid=xgb_grid, nrounds_max=500, nfold=5)
model_treated = xgb_treated_cv['model']
assert_xgb_booster_uses_cuda(model_treated, 'CV-tuned treated XGBoost model')

print('XGBoost Treated best CV AUC:', round(xgb_treated_cv['best_cv_auc'], 4))
print('XGBoost Treated best nrounds:', xgb_treated_cv['best_nrounds'])
print('XGBoost Treated best params:')
print(xgb_treated_cv['best_params'])


---
## TRAIN CONTROL MODEL WITH CV GRID SEARCH
---


In [21]:
xgb_control_cv = fit_xgb_cv_grid(x_matrix=x_control, y=y_control, grid=xgb_grid, nrounds_max=500, nfold=5)
model_control = xgb_control_cv['model']
assert_xgb_booster_uses_cuda(model_control, 'CV-tuned control XGBoost model')

print('XGBoost Control best CV AUC:', round(xgb_control_cv['best_cv_auc'], 4))
print('XGBoost Control best nrounds:', xgb_control_cv['best_nrounds'])
print('XGBoost Control best params:')
print(xgb_control_cv['best_params'])


---
## TEST AUC FOR CV-TUNED XGBOOST MODELS
---


In [22]:
pred_treated_cv_xgb = model_treated.predict(make_dmatrix(x_test.iloc[test_treated_pos]))
auc_treated_cv_xgb = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated_cv_xgb, 'XGBoost Treated CV-tuned test')

pred_control_cv_xgb = model_control.predict(make_dmatrix(x_test.iloc[test_control_pos]))
auc_control_cv_xgb = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control_cv_xgb, 'XGBoost Control CV-tuned test')

print('CV-tuned XGBoost models trained successfully.')
print()


---
## CPU-ONLY GLMNET COMPARISON
---


In [23]:
if RUN_CPU_ONLY_COMPARISON_MODELS:
    print('Training GLMNET comparison models on CPU with sklearn LogisticRegressionCV.')
    print()
    enet_treated = fit_elastic_net(x_treated, y_treated)

    print('Best treated alpha:', enet_treated['best_alpha'])
    print('Best treated lambda:', enet_treated['best_lambda'])
    print('Best treated CV AUC:', round(enet_treated['best_auc'], 4))
    print()

    enet_control = fit_elastic_net(x_control, y_control)

    print('Best control alpha:', enet_control['best_alpha'])
    print('Best control lambda:', enet_control['best_lambda'])
    print('Best control CV AUC:', round(enet_control['best_auc'], 4))
    print()
else:
    enet_treated = None
    enet_control = None
    print('Skipped GLMNET comparison models because RUN_CPU_ONLY_COMPARISON_MODELS is False.')
    print()


---
## OPTIONAL TEST AUC FOR CPU-ONLY GLMNET COMPARISON
---


In [24]:
if enet_treated is not None and enet_control is not None:
    pred_treated_cv_glmnet = enet_treated['best_model'].predict_proba(x_test.iloc[test_treated_pos])[:, 1]
    auc_treated_cv_glmnet = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated_cv_glmnet, 'GLMNET Treated CV-tuned test')

    pred_control_cv_glmnet = enet_control['best_model'].predict_proba(x_test.iloc[test_control_pos])[:, 1]
    auc_control_cv_glmnet = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control_cv_glmnet, 'GLMNET Control CV-tuned test')
else:
    pred_treated_cv_glmnet = None
    pred_control_cv_glmnet = None
    auc_treated_cv_glmnet = np.nan
    auc_control_cv_glmnet = np.nan
    print('Skipped GLMNET test AUC because CPU-only GLMNET training was skipped.')
print()


---
## 15. SCORE TEST SETS
---


In [25]:
p_treated_xgboost = model_treated.predict(make_dmatrix(x_test))
p_control_xgboost = model_control.predict(make_dmatrix(x_test))
results_test_xgboost = build_uplift_results(test_df, p_treated_xgboost, p_control_xgboost)

# Backward-compatible alias for the primary XGBoost result.
results_test = results_test_xgboost
print_highest_benefit(results_test_xgboost, 'XGBoost')

if enet_treated is not None and enet_control is not None:
    p_treated_glmnet = enet_treated['best_model'].predict_proba(x_test)[:, 1]
    p_control_glmnet = enet_control['best_model'].predict_proba(x_test)[:, 1]
    results_test_glmnet = build_uplift_results(test_df, p_treated_glmnet, p_control_glmnet)
    print_highest_benefit(results_test_glmnet, 'GLMNET')
else:
    p_treated_glmnet = None
    p_control_glmnet = None
    results_test_glmnet = None
    print('Skipped GLMNET test scoring because CPU-only GLMNET training was skipped.')
    print()


---
## 16. DECILE SUMMARIES
---


In [26]:
decile_summary_xgboost = summarize_uplift_deciles(results_test_xgboost)

# Backward-compatible alias for the primary XGBoost summary.
decile_summary = decile_summary_xgboost

print('XGBoost decile summary:')
display(decile_summary_xgboost)
print()

if results_test_glmnet is not None:
    decile_summary_glmnet = summarize_uplift_deciles(results_test_glmnet)
    print('GLMNET decile summary:')
    display(decile_summary_glmnet)
    print()
else:
    decile_summary_glmnet = None


---
## WRITE-UP PROBABILITY CALIBRATION AND BRIER SCORES
---


In [27]:
def brier_scores_for_results(results, model_label):
    if results is None:
        return None

    rows = []
    for group_label, treatment_value, pred_col in [
        ('Treated', 1, 'pred_ed_if_treated'),
        ('Control', 0, 'pred_ed_if_control'),
    ]:
        subgroup = results[results['intervention_flag'] == treatment_value].copy()
        if subgroup.empty:
            rows.append({
                'model': model_label,
                'group': group_label,
                'n': 0,
                'observed_ed_rate': np.nan,
                'avg_predicted_ed_rate': np.nan,
                'brier_score': np.nan,
            })
            continue

        rows.append({
            'model': model_label,
            'group': group_label,
            'n': len(subgroup),
            'observed_ed_rate': subgroup['outcome_ed_90d'].mean(),
            'avg_predicted_ed_rate': subgroup[pred_col].mean(),
            'brier_score': brier_score_loss(subgroup['outcome_ed_90d'], subgroup[pred_col]),
        })

    return pd.DataFrame(rows)


def calibration_tables_for_results(results, model_label):
    if results is None:
        return None, None

    calibration_parts = []
    for group_label, treatment_value, pred_col in [
        ('Treated', 1, 'pred_ed_if_treated'),
        ('Control', 0, 'pred_ed_if_control'),
    ]:
        subgroup = results[results['intervention_flag'] == treatment_value].copy()
        if subgroup.empty:
            continue

        subgroup['pred_risk_decile'] = ntile_desc(subgroup[pred_col], 10).to_numpy()
        by_decile = (
            subgroup
            .groupby('pred_risk_decile', as_index=False)
            .agg(
                n=('outcome_ed_90d', 'size'),
                avg_predicted_ed_rate=(pred_col, 'mean'),
                observed_ed_rate=('outcome_ed_90d', 'mean'),
            )
            .sort_values('pred_risk_decile')
        )
        by_decile.insert(0, 'group', group_label)
        by_decile.insert(0, 'model', model_label)
        by_decile['calibration_error'] = by_decile['observed_ed_rate'] - by_decile['avg_predicted_ed_rate']
        by_decile['abs_calibration_error'] = by_decile['calibration_error'].abs()
        calibration_parts.append(by_decile)

    if not calibration_parts:
        return None, None

    calibration_by_decile = pd.concat(calibration_parts, ignore_index=True)
    calibration_summary_rows = []
    for (model_name, group_label), frame in calibration_by_decile.groupby(['model', 'group']):
        calibration_summary_rows.append(
            {
                'model': model_name,
                'group': group_label,
                'n': frame['n'].sum(),
                'mean_abs_calibration_error': frame['abs_calibration_error'].mean(),
                'weighted_mean_abs_calibration_error': np.average(
                    frame['abs_calibration_error'],
                    weights=frame['n'],
                ),
                'max_abs_calibration_error': frame['abs_calibration_error'].max(),
            }
        )
    calibration_summary = pd.DataFrame(calibration_summary_rows)
    return calibration_by_decile, calibration_summary


def save_calibration_plot(calibration_by_decile, folder, model_label):
    if calibration_by_decile is None or calibration_by_decile.empty:
        return

    groups = [group for group in ['Control', 'Treated'] if group in set(calibration_by_decile['group'])]
    fig, axes = plt.subplots(1, len(groups), figsize=(7 * len(groups), 5), sharey=True)
    if len(groups) == 1:
        axes = [axes]

    max_rate = max(
        calibration_by_decile['avg_predicted_ed_rate'].max(),
        calibration_by_decile['observed_ed_rate'].max(),
        0.01,
    )

    for ax, group_label in zip(axes, groups):
        group_df = calibration_by_decile[calibration_by_decile['group'] == group_label].sort_values('pred_risk_decile')
        x_values = np.arange(len(group_df))
        bar_width = 0.38

        ax.bar(
            x_values - bar_width / 2,
            group_df['avg_predicted_ed_rate'],
            width=bar_width,
            label='Avg predicted ED rate',
            color='#4C78A8',
        )
        ax.bar(
            x_values + bar_width / 2,
            group_df['observed_ed_rate'],
            width=bar_width,
            label='Observed ED rate',
            color='#F58518',
        )

        for x_pos, n_value in zip(x_values, group_df['n']):
            ax.text(x_pos, max_rate * 1.03, f'n={int(n_value)}', ha='center', va='bottom', fontsize=8, rotation=90)

        ax.set_title(f'{model_label}: {group_label} Calibration')
        ax.set_xlabel('Predicted Risk Decile: 1 = Highest Predicted Risk')
        ax.set_xticks(x_values)
        ax.set_xticklabels(group_df['pred_risk_decile'].astype(str))
        ax.set_ylim(0, max_rate * 1.22)
        ax.grid(axis='y', alpha=0.25)

    axes[0].set_ylabel('ED Rate')
    axes[-1].legend(loc='upper right')
    fig.suptitle(f'{model_label}: Predicted vs Observed ED Rate by Risk Decile', y=1.03)
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_calibration_plot.png', dpi=150, bbox_inches='tight')
    plt.close(fig)


def save_probability_evaluation_outputs(results, folder, model_label):
    brier_df = brier_scores_for_results(results, model_label)
    calibration_by_decile, calibration_summary = calibration_tables_for_results(results, model_label)

    if brier_df is not None:
        brier_df.to_csv(folder / 'model_brier_scores.csv', index=False)
        print(f'{model_label} Brier scores:')
        display(brier_df)

    if calibration_by_decile is not None:
        calibration_by_decile.to_csv(folder / 'calibration_by_decile.csv', index=False)
        calibration_summary.to_csv(folder / 'calibration_summary.csv', index=False)
        save_calibration_plot(calibration_by_decile, folder, model_label)
        print(f'{model_label} calibration summary:')
        display(calibration_summary)
        print()

    return brier_df, calibration_by_decile, calibration_summary


brier_xgboost, calibration_by_decile_xgboost, calibration_summary_xgboost = save_probability_evaluation_outputs(
    results_test_xgboost,
    xgboost_output_folder,
    'XGBoost',
)

if results_test_glmnet is not None:
    brier_glmnet, calibration_by_decile_glmnet, calibration_summary_glmnet = save_probability_evaluation_outputs(
        results_test_glmnet,
        glmnet_output_folder,
        'GLMNET',
    )
else:
    brier_glmnet = None
    calibration_by_decile_glmnet = None
    calibration_summary_glmnet = None
    print('Skipped GLMNET Brier/calibration outputs because GLMNET scoring was not available.')


---
## WRITE-UP FACTUAL OUTCOME MODEL DIAGNOSTICS
---


In [28]:
def factual_event_count_summary(train_df, test_df):
    rows = []
    for split_name, frame in [('Train', train_df), ('Test', test_df)]:
        for group_label, treatment_value in [('Treated', 1), ('Control', 0)]:
            subgroup = frame[frame['intervention_flag'] == treatment_value]
            n = len(subgroup)
            events = int(subgroup['outcome_ed_90d'].sum()) if n else 0
            rows.append(
                {
                    'split': split_name,
                    'group': group_label,
                    'n': n,
                    'positive_ed_events': events,
                    'negative_ed_events': n - events,
                    'event_rate': events / n if n else np.nan,
                }
            )
    return pd.DataFrame(rows)


def factual_prediction_diagnostics(results, model_label):
    separation_rows = []
    threshold_rows = []
    range_rows = []
    specs = [
        ('Treated', 1, 'pred_ed_if_treated'),
        ('Control', 0, 'pred_ed_if_control'),
    ]

    for group_label, treatment_value, pred_col in specs:
        subgroup = results[results['intervention_flag'] == treatment_value].copy()
        n = len(subgroup)
        events = int(subgroup['outcome_ed_90d'].sum()) if n else 0
        non_events = n - events
        positives = subgroup[subgroup['outcome_ed_90d'] == 1]
        negatives = subgroup[subgroup['outcome_ed_90d'] == 0]
        auc_value = roc_auc_score(subgroup['outcome_ed_90d'], subgroup[pred_col]) if subgroup['outcome_ed_90d'].nunique() == 2 else np.nan

        separation_rows.append(
            {
                'model': model_label,
                'group': group_label,
                'prediction_column': pred_col,
                'n': n,
                'positive_ed_events': events,
                'negative_ed_events': non_events,
                'event_rate': events / n if n else np.nan,
                'auc': auc_value,
                'avg_pred_actual_positive': positives[pred_col].mean() if len(positives) else np.nan,
                'avg_pred_actual_negative': negatives[pred_col].mean() if len(negatives) else np.nan,
                'avg_pred_positive_minus_negative': (
                    positives[pred_col].mean() - negatives[pred_col].mean()
                    if len(positives) and len(negatives)
                    else np.nan
                ),
                'median_pred_actual_positive': positives[pred_col].median() if len(positives) else np.nan,
                'median_pred_actual_negative': negatives[pred_col].median() if len(negatives) else np.nan,
                'median_pred_positive_minus_negative': (
                    positives[pred_col].median() - negatives[pred_col].median()
                    if len(positives) and len(negatives)
                    else np.nan
                ),
            }
        )

        threshold = events / n if n else np.nan
        predicted_positive = subgroup[pred_col] >= threshold if n and not pd.isna(threshold) else pd.Series(False, index=subgroup.index)
        actual_positive = subgroup['outcome_ed_90d'] == 1
        true_positive = int((predicted_positive & actual_positive).sum())
        false_positive = int((predicted_positive & ~actual_positive).sum())
        true_negative = int((~predicted_positive & ~actual_positive).sum())
        false_negative = int((~predicted_positive & actual_positive).sum())
        predicted_positive_n = true_positive + false_positive
        precision = true_positive / predicted_positive_n if predicted_positive_n else np.nan
        recall = true_positive / events if events else np.nan

        threshold_rows.append(
            {
                'model': model_label,
                'group': group_label,
                'prediction_column': pred_col,
                'threshold_rule': 'factual_group_test_event_rate',
                'threshold': threshold,
                'n': n,
                'actual_positive_n': events,
                'actual_negative_n': non_events,
                'predicted_positive_n': predicted_positive_n,
                'predicted_negative_n': true_negative + false_negative,
                'true_positive': true_positive,
                'false_positive': false_positive,
                'true_negative': true_negative,
                'false_negative': false_negative,
                'precision': precision,
                'recall': recall,
            }
        )

        predictions = subgroup[pred_col].dropna()
        range_rows.append(
            {
                'model': model_label,
                'group': group_label,
                'prediction_column': pred_col,
                'n': n,
                'min_pred': predictions.min() if len(predictions) else np.nan,
                'p10_pred': predictions.quantile(0.10) if len(predictions) else np.nan,
                'median_pred': predictions.median() if len(predictions) else np.nan,
                'mean_pred': predictions.mean() if len(predictions) else np.nan,
                'p90_pred': predictions.quantile(0.90) if len(predictions) else np.nan,
                'max_pred': predictions.max() if len(predictions) else np.nan,
            }
        )

    return pd.DataFrame(separation_rows), pd.DataFrame(threshold_rows), pd.DataFrame(range_rows)


event_count_summary = factual_event_count_summary(train_df, test_df)
event_count_summary_path = output_folder / 'factual_event_count_summary.csv'
event_count_summary.to_csv(event_count_summary_path, index=False)

diagnostic_frames = []
threshold_frames = []
range_frames = []

sep_xgboost, threshold_xgboost, range_xgboost = factual_prediction_diagnostics(results_test_xgboost, 'XGBoost')
diagnostic_frames.append(sep_xgboost)
threshold_frames.append(threshold_xgboost)
range_frames.append(range_xgboost)

if results_test_glmnet is not None:
    sep_glmnet, threshold_glmnet, range_glmnet = factual_prediction_diagnostics(results_test_glmnet, 'GLMNET')
    diagnostic_frames.append(sep_glmnet)
    threshold_frames.append(threshold_glmnet)
    range_frames.append(range_glmnet)

factual_prediction_separation = pd.concat(diagnostic_frames, ignore_index=True)
factual_event_rate_threshold_classification = pd.concat(threshold_frames, ignore_index=True)
factual_prediction_ranges = pd.concat(range_frames, ignore_index=True)

factual_prediction_separation_path = output_folder / 'factual_prediction_separation.csv'
factual_event_rate_threshold_classification_path = output_folder / 'factual_event_rate_threshold_classification.csv'
factual_prediction_ranges_path = output_folder / 'factual_prediction_ranges.csv'

factual_prediction_separation.to_csv(factual_prediction_separation_path, index=False)
factual_event_rate_threshold_classification.to_csv(factual_event_rate_threshold_classification_path, index=False)
factual_prediction_ranges.to_csv(factual_prediction_ranges_path, index=False)

print('Factual train/test event counts:')
display(event_count_summary)
print('Factual positive-vs-negative prediction separation:')
display(factual_prediction_separation)
print('Event-rate threshold classification:')
display(factual_event_rate_threshold_classification)
print('Prediction ranges:')
display(factual_prediction_ranges)
print('Factual diagnostic outputs written to:', output_folder)


---
## WRITE-UP OBSERVED TREATED-CONTROL GAP BY UPLIFT DECILE
---


In [29]:
def wilson_score_interval(events, n, confidence=0.95):
    """Wilson score confidence interval for one binomial proportion."""
    if n == 0:
        return np.nan, np.nan

    z_lookup = {0.90: 1.644854, 0.95: 1.959964, 0.99: 2.575829}
    z_value = z_lookup.get(confidence, 1.959964)
    p_hat = events / n
    denominator = 1 + (z_value ** 2 / n)
    center = (p_hat + (z_value ** 2 / (2 * n))) / denominator
    half_width = (
        z_value
        * np.sqrt((p_hat * (1 - p_hat) / n) + (z_value ** 2 / (4 * n ** 2)))
        / denominator
    )
    return max(0, center - half_width), min(1, center + half_width)


def difference_in_proportions_ci(control_events, control_n, treated_events, treated_n, confidence=0.95):
    """Newcombe/Wilson CI for control observed ED rate minus treated observed ED rate."""
    if control_n == 0 or treated_n == 0:
        return np.nan, np.nan, np.nan

    control_rate = control_events / control_n
    treated_rate = treated_events / treated_n
    gap = control_rate - treated_rate
    control_lower, control_upper = wilson_score_interval(control_events, control_n, confidence)
    treated_lower, treated_upper = wilson_score_interval(treated_events, treated_n, confidence)
    gap_lower = control_lower - treated_upper
    gap_upper = control_upper - treated_lower
    return gap, gap_lower, gap_upper


def observed_gap_by_decile(results, model_label):
    if results is None:
        return None

    rows = []
    for decile, decile_df in results.groupby('uplift_decile'):
        treated = decile_df[decile_df['intervention_flag'] == 1]
        control = decile_df[decile_df['intervention_flag'] == 0]
        treated_events = int(treated['outcome_ed_90d'].sum()) if not treated.empty else 0
        control_events = int(control['outcome_ed_90d'].sum()) if not control.empty else 0
        treated_rate = treated_events / len(treated) if len(treated) else np.nan
        control_rate = control_events / len(control) if len(control) else np.nan
        gap, gap_ci_lower, gap_ci_upper = difference_in_proportions_ci(
            control_events,
            len(control),
            treated_events,
            len(treated),
        )
        rows.append({
            'model': model_label,
            'uplift_decile': decile,
            'n': len(decile_df),
            'treated_n': len(treated),
            'control_n': len(control),
            'treated_events': treated_events,
            'control_events': control_events,
            'observed_ed_rate': decile_df['outcome_ed_90d'].mean(),
            'treated_observed_ed_rate': treated_rate,
            'control_observed_ed_rate': control_rate,
            'observed_control_minus_treated_gap': gap,
            'observed_gap_ci_lower_95': gap_ci_lower,
            'observed_gap_ci_upper_95': gap_ci_upper,
            'avg_predicted_benefit': decile_df['benefit_score'].mean(),
            'predicted_benefit_within_observed_gap_ci_95': (
                bool(gap_ci_lower <= decile_df['benefit_score'].mean() <= gap_ci_upper)
                if pd.notna(gap_ci_lower) else np.nan
            ),
            'predicted_minus_observed_gap': decile_df['benefit_score'].mean() - gap if pd.notna(gap) else np.nan,
            'abs_predicted_minus_observed_gap': abs(decile_df['benefit_score'].mean() - gap) if pd.notna(gap) else np.nan,
            'avg_pred_ed_if_treated': decile_df['pred_ed_if_treated'].mean(),
            'avg_pred_ed_if_control': decile_df['pred_ed_if_control'].mean(),
            'treated_pct': decile_df['intervention_flag'].mean(),
            'ci_method': 'newcombe_wilson_difference_in_proportions',
        })

    return pd.DataFrame(rows).sort_values('uplift_decile').reset_index(drop=True)


def uplift_curve_by_decile(results, model_label):
    if results is None:
        return None

    rows = []
    sorted_deciles = sorted(results['uplift_decile'].dropna().unique())
    total_n = len(results)
    for max_decile in sorted_deciles:
        curve_df = results[results['uplift_decile'] <= max_decile].copy()
        treated = curve_df[curve_df['intervention_flag'] == 1]
        control = curve_df[curve_df['intervention_flag'] == 0]
        treated_events = int(treated['outcome_ed_90d'].sum()) if not treated.empty else 0
        control_events = int(control['outcome_ed_90d'].sum()) if not control.empty else 0
        gap, gap_ci_lower, gap_ci_upper = difference_in_proportions_ci(
            control_events,
            len(control),
            treated_events,
            len(treated),
        )
        rows.append({
            'model': model_label,
            'through_uplift_decile': max_decile,
            'population_fraction_targeted': len(curve_df) / total_n if total_n else np.nan,
            'n': len(curve_df),
            'treated_n': len(treated),
            'control_n': len(control),
            'treated_events': treated_events,
            'control_events': control_events,
            'avg_predicted_benefit': curve_df['benefit_score'].mean(),
            'observed_control_minus_treated_gap': gap,
            'observed_gap_ci_lower_95': gap_ci_lower,
            'observed_gap_ci_upper_95': gap_ci_upper,
            'predicted_benefit_within_observed_gap_ci_95': (
                bool(gap_ci_lower <= curve_df['benefit_score'].mean() <= gap_ci_upper)
                if pd.notna(gap_ci_lower) else np.nan
            ),
        })

    curve_df = pd.DataFrame(rows).sort_values('through_uplift_decile').reset_index(drop=True)
    return curve_df


def uplift_curve_summary(curve_df, model_label):
    if curve_df is None or curve_df.empty:
        return None

    valid_observed = curve_df[['population_fraction_targeted', 'observed_control_minus_treated_gap']].dropna()
    valid_predicted = curve_df[['population_fraction_targeted', 'avg_predicted_benefit']].dropna()
    observed_area = np.trapz(
        valid_observed['observed_control_minus_treated_gap'],
        valid_observed['population_fraction_targeted'],
    ) if len(valid_observed) >= 2 else np.nan
    predicted_area = np.trapz(
        valid_predicted['avg_predicted_benefit'],
        valid_predicted['population_fraction_targeted'],
    ) if len(valid_predicted) >= 2 else np.nan

    return pd.DataFrame([
        {
            'model': model_label,
            'observed_gap_area_under_curve': observed_area,
            'predicted_benefit_area_under_curve': predicted_area,
            'final_cumulative_observed_gap': curve_df['observed_control_minus_treated_gap'].iloc[-1],
            'final_cumulative_avg_predicted_benefit': curve_df['avg_predicted_benefit'].iloc[-1],
        }
    ])


def save_observed_gap_outputs(results, folder, model_label):
    gap_df = observed_gap_by_decile(results, model_label)
    if gap_df is None:
        return None, None, None

    gap_df.to_csv(folder / 'uplift_observed_gap_by_decile.csv', index=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    x_values = np.arange(len(gap_df))
    gaps = gap_df['observed_control_minus_treated_gap'].to_numpy(dtype=float)
    ci_lower = gap_df['observed_gap_ci_lower_95'].to_numpy(dtype=float)
    ci_upper = gap_df['observed_gap_ci_upper_95'].to_numpy(dtype=float)
    yerr = np.vstack([gaps - ci_lower, ci_upper - gaps])
    ax.bar(x_values, gaps, color='#4c78a8')
    ax.errorbar(x_values, gaps, yerr=yerr, fmt='none', ecolor='#222222', capsize=4, linewidth=1)
    ax.axhline(0, color='gray', linewidth=1)
    ax.set_xticks(x_values)
    ax.set_xticklabels(gap_df['uplift_decile'].astype(str))
    ax.set_title(f'{model_label}: Observed Control-Treated ED Gap by Uplift Decile')
    ax.set_xlabel('Uplift Decile: 1 = Highest Predicted Benefit')
    ax.set_ylabel('Observed Control ED Rate - Treated ED Rate')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_observed_gap_by_decile.png', dpi=150)
    plt.close(fig)

    curve_df = uplift_curve_by_decile(results, model_label)
    curve_summary = uplift_curve_summary(curve_df, model_label)
    if curve_df is not None:
        curve_df.to_csv(folder / 'uplift_curve_by_decile.csv', index=False)

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(
            curve_df['population_fraction_targeted'],
            curve_df['avg_predicted_benefit'],
            marker='o',
            label='Average predicted benefit',
        )
        ax.plot(
            curve_df['population_fraction_targeted'],
            curve_df['observed_control_minus_treated_gap'],
            marker='o',
            label='Observed control-treated gap',
        )
        ax.fill_between(
            curve_df['population_fraction_targeted'].to_numpy(dtype=float),
            curve_df['observed_gap_ci_lower_95'].to_numpy(dtype=float),
            curve_df['observed_gap_ci_upper_95'].to_numpy(dtype=float),
            color='#f58518',
            alpha=0.18,
            label='Observed gap 95% CI',
        )
        ax.axhline(0, color='gray', linewidth=1)
        ax.set_title(f'{model_label}: Cumulative Uplift Curve by Targeted Fraction')
        ax.set_xlabel('Cumulative Fraction Targeted')
        ax.set_ylabel('ED Rate Difference / Predicted Benefit')
        ax.legend()
        fig.tight_layout()
        fig.savefig(folder / 'dashboard_uplift_curve_by_decile.png', dpi=150)
        plt.close(fig)

    if curve_summary is not None:
        curve_summary.to_csv(folder / 'uplift_curve_summary.csv', index=False)

    print(f'{model_label} observed control-treated gap by uplift decile:')
    display(gap_df)
    print()
    print(f'{model_label} cumulative uplift curve by decile:')
    display(curve_df)
    print()
    if curve_summary is not None:
        print(f'{model_label} uplift curve summary:')
        display(curve_summary)
        print()
    return gap_df, curve_df, curve_summary


observed_gap_xgboost, uplift_curve_xgboost, uplift_curve_summary_xgboost = save_observed_gap_outputs(
    results_test_xgboost,
    xgboost_output_folder,
    'XGBoost',
)

if results_test_glmnet is not None:
    observed_gap_glmnet, uplift_curve_glmnet, uplift_curve_summary_glmnet = save_observed_gap_outputs(
        results_test_glmnet,
        glmnet_output_folder,
        'GLMNET',
    )
else:
    observed_gap_glmnet = None
    uplift_curve_glmnet = None
    uplift_curve_summary_glmnet = None
    print('Skipped GLMNET observed gap outputs because GLMNET scoring was not available.')




---
## 17. VARIABLE IMPORTANCE
---


In [30]:
importance_treated_xgboost = xgb_importance_frame(model_treated)
importance_control_xgboost = xgb_importance_frame(model_control)

# Backward-compatible aliases for the primary XGBoost importance tables.
importance_treated = importance_treated_xgboost
importance_control = importance_control_xgboost

print('Top variables in XGBoost treated model:')
display(importance_treated_xgboost.head(20))
print()

print('Top variables in XGBoost control model:')
display(importance_control_xgboost.head(20))
print()

if enet_treated is not None and enet_control is not None:
    glmnet_importance_treated = glmnet_contribution_importance_frame(enet_treated, x_test, 'Treated Model')
    glmnet_importance_control = glmnet_contribution_importance_frame(enet_control, x_test, 'Control Model')

    print('Top variables in GLMNET treated model:')
    display(glmnet_importance_treated.head(20))
    print()

    print('Top variables in GLMNET control model:')
    display(glmnet_importance_control.head(20))
    print()
else:
    glmnet_importance_treated = None
    glmnet_importance_control = None


---
## 18. SCORE FULL FILES
---


In [31]:
full_x_df = model_df[feature_cols].copy()
_, [full_matrix_raw] = make_design_matrix([full_x_df])
full_matrix = align_to_columns(full_matrix_raw, combined_matrix.columns)

full_pred_treated_xgboost = model_treated.predict(make_dmatrix(full_matrix))
full_pred_control_xgboost = model_control.predict(make_dmatrix(full_matrix))
scored_full_xgboost = build_uplift_results(model_df, full_pred_treated_xgboost, full_pred_control_xgboost)

# Backward-compatible alias for the primary XGBoost scored file.
scored_full = scored_full_xgboost

if enet_treated is not None and enet_control is not None:
    full_pred_treated_glmnet = enet_treated['best_model'].predict_proba(full_matrix)[:, 1]
    full_pred_control_glmnet = enet_control['best_model'].predict_proba(full_matrix)[:, 1]
    scored_full_glmnet = build_uplift_results(model_df, full_pred_treated_glmnet, full_pred_control_glmnet)
else:
    full_pred_treated_glmnet = None
    full_pred_control_glmnet = None
    scored_full_glmnet = None


---
## 19. WRITE OUTPUTS
---


In [32]:
scored_full_xgboost.to_csv(xgboost_output_path, index=False)
decile_summary_xgboost.to_csv(xgboost_summary_path, index=False)

print('XGBoost scored full file written to:', xgboost_output_path, '\n')
print('XGBoost decile summary written to:', xgboost_summary_path, '\n')

if scored_full_glmnet is not None and decile_summary_glmnet is not None:
    scored_full_glmnet.to_csv(glmnet_output_path, index=False)
    decile_summary_glmnet.to_csv(glmnet_summary_path, index=False)

    print('GLMNET scored full file written to:', glmnet_output_path, '\n')
    print('GLMNET decile summary written to:', glmnet_summary_path, '\n')
else:
    print('Skipped GLMNET core CSV outputs because GLMNET scoring was not available.\n')


---
## 20. INTERPRETATION
---


In [33]:
print('INTERPRETATION:')
print('- pred_ed_if_treated = predicted probability of ED within 90d if treated')
print('- pred_ed_if_control = predicted probability of ED within 90d if not treated')
print('- benefit_score = pred_ed_if_control - pred_ed_if_treated')
print('- Higher benefit_score means treatment is predicted to reduce ED risk more')
print('- Uplift decile 1 = highest predicted treatment benefit')


---
## DASHBOARD VIEWS
---


In [34]:
dashboard_folder = xgboost_output_folder


def save_bar_chart(df, x_col, y_col, title, x_label, y_label, path, width=8, height=5):
    fig, ax = plt.subplots(figsize=(width, height))
    ax.bar(df[x_col].astype(str), df[y_col])
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def save_decile_dashboard_charts(decile_df, folder, model_label):
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'avg_benefit_score',
        f'{model_label}: Average Predicted Intervention Benefit by Uplift Decile',
        'Uplift Decile: 1 = Highest Predicted Benefit',
        'Average Benefit Score',
        folder / 'dashboard_avg_benefit_by_decile.png',
    )
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'observed_ed_rate',
        f'{model_label}: Observed 90-Day ED Rate by Uplift Decile',
        'Uplift Decile',
        'Observed ED Rate',
        folder / 'dashboard_observed_ed_rate_by_decile.png',
    )
    save_bar_chart(
        decile_df,
        'uplift_decile',
        'treated_pct',
        f'{model_label}: Current Treatment Penetration by Uplift Decile',
        'Uplift Decile',
        'Percent Treated',
        folder / 'dashboard_treated_pct_by_decile.png',
    )

    decile_long = decile_df.melt(
        id_vars='uplift_decile',
        value_vars=['avg_pred_ed_if_treated', 'avg_pred_ed_if_control'],
        var_name='scenario',
        value_name='predicted_ed_rate',
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    scenarios = list(decile_long['scenario'].unique())
    x_values = np.arange(len(decile_df))
    bar_width = 0.35
    for offset, scenario in enumerate(scenarios):
        values = decile_long[decile_long['scenario'] == scenario]['predicted_ed_rate'].to_numpy()
        ax.bar(x_values + (offset - 0.5) * bar_width, values, width=bar_width, label=scenario)
    ax.set_xticks(x_values)
    ax.set_xticklabels(decile_df['uplift_decile'].astype(str))
    ax.set_title(f'{model_label}: Predicted ED Risk Treated vs Control by Decile')
    ax.set_xlabel('Uplift Decile')
    ax.set_ylabel('Predicted ED Rate')
    ax.legend()
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_predicted_treated_vs_control.png', dpi=150)
    plt.close(fig)


save_decile_dashboard_charts(decile_summary_xgboost, xgboost_output_folder, 'XGBoost')
print('XGBoost dashboard charts saved to:', xgboost_output_folder)

if decile_summary_glmnet is not None:
    save_decile_dashboard_charts(decile_summary_glmnet, glmnet_output_folder, 'GLMNET')
    print('GLMNET dashboard charts saved to:', glmnet_output_folder)
else:
    print('Skipped GLMNET dashboard charts because GLMNET scoring was not available.')


---
## ROI PER DECILE
---


In [35]:
cost_per_ed_visit = 1200
cost_per_intervention = 250


def build_roi_summary(decile_df):
    roi_df = decile_df.copy()
    roi_df['expected_ed_rate_reduction'] = roi_df['avg_benefit_score']
    roi_df['expected_ed_visits_avoided'] = roi_df['n'] * roi_df['expected_ed_rate_reduction']
    roi_df['gross_savings'] = roi_df['expected_ed_visits_avoided'] * cost_per_ed_visit
    roi_df['intervention_cost'] = roi_df['n'] * cost_per_intervention
    roi_df['net_savings'] = roi_df['gross_savings'] - roi_df['intervention_cost']
    roi_df['roi'] = roi_df['net_savings'] / roi_df['intervention_cost']
    return roi_df


def save_roi_outputs(decile_df, folder, model_label):
    roi_df = build_roi_summary(decile_df)
    print(f'{model_label} ROI summary:')
    display(roi_df)
    roi_df.to_csv(folder / 'uplift_roi_by_decile.csv', index=False)
    save_bar_chart(
        roi_df,
        'uplift_decile',
        'net_savings',
        f'{model_label}: Estimated Net Savings by Uplift Decile',
        'Uplift Decile',
        'Estimated Net Savings',
        folder / 'dashboard_roi_net_savings_by_decile.png',
    )
    print(f'{model_label} ROI summary saved to:', folder)
    print()
    return roi_df


roi_summary_xgboost = save_roi_outputs(decile_summary_xgboost, xgboost_output_folder, 'XGBoost')

# Backward-compatible alias for the primary XGBoost ROI summary.
roi_summary = roi_summary_xgboost

if decile_summary_glmnet is not None:
    roi_summary_glmnet = save_roi_outputs(decile_summary_glmnet, glmnet_output_folder, 'GLMNET')
else:
    roi_summary_glmnet = None
    print('Skipped GLMNET ROI outputs because GLMNET scoring was not available.')


---
## WRITE-UP TOP BENEFIT DECILE SUMMARY
---


In [36]:
def top_benefit_decile_summary(results, roi_df, model_label):
    if results is None or roi_df is None:
        return None

    top_decile = results[results['uplift_decile'] == 1].copy()
    if top_decile.empty:
        return None

    treated = top_decile[top_decile['intervention_flag'] == 1]
    control = top_decile[top_decile['intervention_flag'] == 0]
    treated_rate = treated['outcome_ed_90d'].mean() if not treated.empty else np.nan
    control_rate = control['outcome_ed_90d'].mean() if not control.empty else np.nan
    top_roi = roi_df.loc[roi_df['uplift_decile'] == 1].iloc[0]

    return pd.DataFrame(
        [
            {
                'model': model_label,
                'top_decile_n': len(top_decile),
                'top_decile_treated_n': len(treated),
                'top_decile_control_n': len(control),
                'top_decile_avg_predicted_benefit': top_decile['benefit_score'].mean(),
                'top_decile_observed_ed_rate': top_decile['outcome_ed_90d'].mean(),
                'top_decile_treated_observed_ed_rate': treated_rate,
                'top_decile_control_observed_ed_rate': control_rate,
                'top_decile_observed_control_minus_treated_gap': control_rate - treated_rate,
                'top_decile_treated_pct': top_decile['intervention_flag'].mean(),
                'top_decile_estimated_ed_visits_avoided': top_roi['expected_ed_visits_avoided'],
                'top_decile_gross_savings': top_roi['gross_savings'],
                'top_decile_intervention_cost': top_roi['intervention_cost'],
                'top_decile_net_savings': top_roi['net_savings'],
                'top_decile_roi': top_roi['roi'],
            }
        ]
    )


def save_top_decile_summary(results, roi_df, folder, model_label):
    summary = top_benefit_decile_summary(results, roi_df, model_label)
    if summary is None:
        return None
    summary.to_csv(folder / 'top_benefit_decile_summary.csv', index=False)
    print(f'{model_label} top benefit decile summary:')
    display(summary)
    print()
    return summary


top_decile_summary_xgboost = save_top_decile_summary(
    results_test_xgboost,
    roi_summary_xgboost,
    xgboost_output_folder,
    'XGBoost',
)

if results_test_glmnet is not None and roi_summary_glmnet is not None:
    top_decile_summary_glmnet = save_top_decile_summary(
        results_test_glmnet,
        roi_summary_glmnet,
        glmnet_output_folder,
        'GLMNET',
    )
else:
    top_decile_summary_glmnet = None
    print('Skipped GLMNET top benefit decile summary because GLMNET scoring or ROI was not available.')


---
## RISK MODEL DRIVER OUTPUTS
---


In [37]:
def save_model_driver_outputs(
    treated_importance,
    control_importance,
    folder,
    model_label,
    value_col,
    x_label,
    csv_name='shap_importance_treated_control_models.csv',
):
    combined = pd.concat([treated_importance, control_importance], ignore_index=True)
    combined.to_csv(folder / csv_name, index=False)

    for driver_df, treatment_label, filename in [
        (treated_importance, 'Treated Model', 'dashboard_shap_treated_model.png'),
        (control_importance, 'Control Model', 'dashboard_shap_control_model.png'),
    ]:
        top = driver_df.nlargest(20, value_col).sort_values(value_col)
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(top['feature'], top[value_col])
        ax.set_title(f'{model_label}: Top Risk Drivers - {treatment_label}')
        ax.set_xlabel(x_label)
        ax.set_ylabel('Feature')
        fig.tight_layout()
        fig.savefig(folder / filename, dpi=150)
        plt.close(fig)

    print(f'{model_label} risk-driver outputs saved to:', folder)
    return combined


shap_treated_importance = shap_importance_frame(model_treated, x_test, 'Treated Model')
shap_control_importance = shap_importance_frame(model_control, x_test, 'Control Model')
shap_importance_combined = save_model_driver_outputs(
    shap_treated_importance,
    shap_control_importance,
    xgboost_output_folder,
    'XGBoost',
    'mean_abs_shap',
    'Mean Absolute SHAP Contribution',
)

if enet_treated is not None and enet_control is not None:
    # GLMNET does not use XGBoost SHAP contributions here, so save an analogous
    # standardized logit-contribution driver artifact in the GLMNET folder.
    glmnet_driver_treated = glmnet_contribution_importance_frame(enet_treated, x_test, 'Treated Model')
    glmnet_driver_control = glmnet_contribution_importance_frame(enet_control, x_test, 'Control Model')
    glmnet_driver_combined = save_model_driver_outputs(
        glmnet_driver_treated,
        glmnet_driver_control,
        glmnet_output_folder,
        'GLMNET',
        'mean_abs_model_contribution',
        'Mean Absolute Standardized Logit Contribution',
    )
else:
    glmnet_driver_treated = None
    glmnet_driver_control = None
    glmnet_driver_combined = None
    print('Skipped GLMNET model-driver outputs because GLMNET scoring was not available.')

print('\nFiles currently in XGBoost output folder:')
print('\n'.join(str(path) for path in sorted(xgboost_output_folder.iterdir())))

print('\nFiles currently in GLMNET output folder:')
print('\n'.join(str(path) for path in sorted(glmnet_output_folder.iterdir())))


---
## BENEFIT SCORE DRIVER IMPORTANCE
---


In [38]:
def save_benefit_driver_outputs(importance_df, folder, model_label, value_col, x_label):
    importance_df.to_csv(
        folder / 'shap_importance_benefit_score.csv',
        index=False,
    )

    top = importance_df.head(20).sort_values(value_col)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top[value_col])
    ax.set_title(f'{model_label}: Top Drivers of Predicted Treatment Benefit')
    ax.set_xlabel(x_label)
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(folder / 'dashboard_shap_benefit_score.png', dpi=150)
    plt.close(fig)

    print(f'{model_label} benefit-driver outputs saved to:', folder)
    print()


def xgboost_benefit_shap_importance_frame(model_treated, model_control, x_matrix):
    benefit_dtest = make_dmatrix(x_matrix)

    control_contribs = model_control.predict(benefit_dtest, pred_contribs=True)
    treated_contribs = model_treated.predict(benefit_dtest, pred_contribs=True)
    shap_columns = [*x_matrix.columns, 'BIAS']

    control_shap = pd.DataFrame(control_contribs, columns=shap_columns, index=x_matrix.index)
    treated_shap = pd.DataFrame(treated_contribs, columns=shap_columns, index=x_matrix.index)

    # Signed contribution to model-estimated benefit on the XGBoost raw-margin scale.
    benefit_shap = control_shap - treated_shap
    benefit_shap_no_bias = benefit_shap.drop(columns=['BIAS'], errors='ignore')

    return (
        pd.DataFrame(
            {
                'feature': benefit_shap_no_bias.columns,
                'mean_abs_benefit_shap': benefit_shap_no_bias.abs().mean(axis=0).to_numpy(),
                'mean_signed_benefit_shap': benefit_shap_no_bias.mean(axis=0).to_numpy(),
                'pct_positive_benefit_shap': benefit_shap_no_bias.gt(0).mean(axis=0).to_numpy(),
                'importance_type': 'xgboost_raw_margin_shap_difference',
            }
        )
        .sort_values('mean_abs_benefit_shap', ascending=False)
        .reset_index(drop=True)
    )


def glmnet_benefit_contribution_importance_frame(treated_model_info, control_model_info, x_matrix):
    treated_pipeline = treated_model_info['best_model']
    control_pipeline = control_model_info['best_model']

    treated_scaler = treated_pipeline.named_steps['standardscaler']
    control_scaler = control_pipeline.named_steps['standardscaler']
    treated_model = treated_pipeline.named_steps['logisticregressioncv']
    control_model = control_pipeline.named_steps['logisticregressioncv']

    treated_contrib = pd.DataFrame(
        treated_scaler.transform(x_matrix) * treated_model.coef_.ravel(),
        columns=x_matrix.columns,
        index=x_matrix.index,
    )
    control_contrib = pd.DataFrame(
        control_scaler.transform(x_matrix) * control_model.coef_.ravel(),
        columns=x_matrix.columns,
        index=x_matrix.index,
    )

    # Signed contribution to model-estimated benefit on the GLMNET logit scale.
    benefit_contrib = control_contrib - treated_contrib

    return (
        pd.DataFrame(
            {
                'feature': benefit_contrib.columns,
                'mean_abs_benefit_contribution': benefit_contrib.abs().mean(axis=0).to_numpy(),
                'mean_signed_benefit_contribution': benefit_contrib.mean(axis=0).to_numpy(),
                'pct_positive_benefit_contribution': benefit_contrib.gt(0).mean(axis=0).to_numpy(),
                'control_coefficient': control_model.coef_.ravel(),
                'treated_coefficient': treated_model.coef_.ravel(),
                'coefficient_difference': control_model.coef_.ravel() - treated_model.coef_.ravel(),
                'importance_type': 'glmnet_standardized_logit_contribution_difference',
            }
        )
        .sort_values('mean_abs_benefit_contribution', ascending=False)
        .reset_index(drop=True)
    )


xgboost_benefit_shap_importance = xgboost_benefit_shap_importance_frame(
    model_treated,
    model_control,
    x_test,
)

print('Top XGBoost SHAP features driving predicted treatment benefit:')
display(xgboost_benefit_shap_importance.head(20))

save_benefit_driver_outputs(
    xgboost_benefit_shap_importance,
    xgboost_output_folder,
    'XGBoost',
    'mean_abs_benefit_shap',
    'Mean Absolute SHAP Contribution to Benefit',
)

if enet_treated is not None and enet_control is not None:
    glmnet_benefit_importance = glmnet_benefit_contribution_importance_frame(
        enet_treated,
        enet_control,
        x_test,
    )

    print('Top GLMNET features driving predicted treatment benefit:')
    display(glmnet_benefit_importance.head(20))

    save_benefit_driver_outputs(
        glmnet_benefit_importance,
        glmnet_output_folder,
        'GLMNET',
        'mean_abs_benefit_contribution',
        'Mean Absolute Standardized Logit Contribution to Benefit',
    )
else:
    glmnet_benefit_importance = None
    print('Skipped GLMNET benefit-driver outputs because GLMNET scoring was not available.')


---
## WRITE-UP MODEL EVALUATION SUMMARY
---


In [39]:
def get_group_metric(df, group, metric_col):
    if df is None:
        return np.nan
    match = df[df['group'] == group]
    if match.empty:
        return np.nan
    return float(match[metric_col].iloc[0])


def get_top_gap_metric(gap_df, metric_col):
    if gap_df is None or gap_df.empty:
        return np.nan
    match = gap_df[gap_df['uplift_decile'] == 1]
    if match.empty:
        return np.nan
    return match[metric_col].iloc[0]


def get_curve_metric(curve_summary_df, metric_col):
    if curve_summary_df is None or curve_summary_df.empty:
        return np.nan
    return float(curve_summary_df[metric_col].iloc[0])


def decile_gap_rank_correlation(gap_df):
    if gap_df is None or gap_df.empty:
        return np.nan
    valid = gap_df[['avg_predicted_benefit', 'observed_control_minus_treated_gap']].dropna()
    if len(valid) < 2:
        return np.nan
    return valid['avg_predicted_benefit'].corr(valid['observed_control_minus_treated_gap'], method='spearman')


def mean_abs_predicted_observed_gap_difference(gap_df):
    if gap_df is None or gap_df.empty:
        return np.nan
    return float(gap_df['abs_predicted_minus_observed_gap'].mean())


def build_model_evaluation_row(model_label, treated_cv_auc, control_cv_auc, treated_test_auc, control_test_auc, brier_df, calibration_summary, gap_df, curve_summary_df):
    return {
        'model': model_label,
        'treated_cv_auc': treated_cv_auc,
        'control_cv_auc': control_cv_auc,
        'treated_test_auc': treated_test_auc,
        'control_test_auc': control_test_auc,
        'treated_brier_score': get_group_metric(brier_df, 'Treated', 'brier_score'),
        'control_brier_score': get_group_metric(brier_df, 'Control', 'brier_score'),
        'treated_calibration_error': get_group_metric(calibration_summary, 'Treated', 'weighted_mean_abs_calibration_error'),
        'control_calibration_error': get_group_metric(calibration_summary, 'Control', 'weighted_mean_abs_calibration_error'),
        'top_decile_avg_predicted_benefit': get_top_gap_metric(gap_df, 'avg_predicted_benefit'),
        'top_decile_observed_control_minus_treated_gap': get_top_gap_metric(gap_df, 'observed_control_minus_treated_gap'),
        'top_decile_observed_gap_ci_lower_95': get_top_gap_metric(gap_df, 'observed_gap_ci_lower_95'),
        'top_decile_observed_gap_ci_upper_95': get_top_gap_metric(gap_df, 'observed_gap_ci_upper_95'),
        'top_decile_predicted_benefit_within_observed_gap_ci_95': get_top_gap_metric(gap_df, 'predicted_benefit_within_observed_gap_ci_95'),
        'benefit_gap_spearman_corr_by_decile': decile_gap_rank_correlation(gap_df),
        'mean_abs_predicted_minus_observed_gap_by_decile': mean_abs_predicted_observed_gap_difference(gap_df),
        'observed_gap_area_under_curve': get_curve_metric(curve_summary_df, 'observed_gap_area_under_curve'),
        'predicted_benefit_area_under_curve': get_curve_metric(curve_summary_df, 'predicted_benefit_area_under_curve'),
    }


model_evaluation_rows = [
    build_model_evaluation_row(
        'XGBoost',
        xgb_treated_cv['best_cv_auc'],
        xgb_control_cv['best_cv_auc'],
        auc_treated_cv_xgb,
        auc_control_cv_xgb,
        brier_xgboost,
        calibration_summary_xgboost,
        observed_gap_xgboost,
        uplift_curve_summary_xgboost,
    )
]

if enet_treated is not None and enet_control is not None:
    model_evaluation_rows.append(
        build_model_evaluation_row(
            'GLMNET',
            enet_treated['best_auc'],
            enet_control['best_auc'],
            auc_treated_cv_glmnet,
            auc_control_cv_glmnet,
            brier_glmnet,
            calibration_summary_glmnet,
            observed_gap_glmnet,
            uplift_curve_summary_glmnet,
        )
    )

model_evaluation_summary = pd.DataFrame(model_evaluation_rows)
model_evaluation_summary_path = output_folder / 'model_evaluation_summary.csv'
model_evaluation_summary.to_csv(model_evaluation_summary_path, index=False)

print('Model evaluation summary:')
display(model_evaluation_summary)
print('Model evaluation summary written to:', model_evaluation_summary_path)




---
## WRITE-UP MODEL RECOMMENDATION SUMMARY
---


In [40]:
def auc_strength(value):
    if pd.isna(value):
        return 'Unavailable'
    if value >= 0.90:
        return 'Excellent'
    if value >= 0.80:
        return 'Strong'
    if value >= 0.70:
        return 'Acceptable'
    if value >= 0.60:
        return 'Weak'
    return 'Poor'


def calibration_label(row):
    avg_error = np.nanmean([row['treated_calibration_error'], row['control_calibration_error']])
    avg_brier = np.nanmean([row['treated_brier_score'], row['control_brier_score']])
    if pd.isna(avg_error):
        return 'Unavailable'
    if avg_error <= 0.02:
        return f'Well calibrated; avg abs calibration error {avg_error:.3f}, avg Brier {avg_brier:.3f}'
    if avg_error <= 0.05:
        return f'Moderately calibrated; avg abs calibration error {avg_error:.3f}, avg Brier {avg_brier:.3f}'
    return f'Calibration needs review; avg abs calibration error {avg_error:.3f}, avg Brier {avg_brier:.3f}'


def benefit_ranking_label(row):
    benefit = row['top_decile_avg_predicted_benefit']
    gap = row['top_decile_observed_control_minus_treated_gap']
    ci_lower = row['top_decile_observed_gap_ci_lower_95']
    ci_upper = row['top_decile_observed_gap_ci_upper_95']
    benefit_within_ci = row['top_decile_predicted_benefit_within_observed_gap_ci_95']
    rank_corr = row['benefit_gap_spearman_corr_by_decile']
    if pd.isna(benefit):
        return 'Unavailable'
    ci_text = f'95% CI {ci_lower:.3f} to {ci_upper:.3f}' if pd.notna(ci_lower) else '95% CI unavailable'
    if pd.isna(benefit_within_ci):
        within_text = 'predicted benefit CI check unavailable'
    elif bool(benefit_within_ci):
        within_text = 'predicted benefit is within observed-gap CI'
    else:
        within_text = 'predicted benefit is outside observed-gap CI'
    corr_text = f'; decile rank corr {rank_corr:.3f}' if pd.notna(rank_corr) else ''
    if benefit > 0 and gap > 0:
        return f'Top decile predicted benefit {benefit:.3f}; observed gap {gap:.3f} ({ci_text}); {within_text}{corr_text}'
    if benefit > 0:
        return f'Top decile predicted benefit {benefit:.3f}; observed gap not positive ({gap:.3f}, {ci_text}); {within_text}{corr_text}'
    return f'Top decile predicted benefit is not positive ({benefit:.3f}); observed gap {gap:.3f} ({ci_text}); {within_text}{corr_text}'


recommendation_rows = []
for _, row in model_evaluation_summary.iterrows():
    avg_auc = np.nanmean([row['treated_test_auc'], row['control_test_auc']])
    recommendation_rows.append(
        {
            'model': row['model'],
            'risk_prediction_strength': f'{auc_strength(avg_auc)} average test AUC ({avg_auc:.3f})',
            'probability_calibration': calibration_label(row),
            'benefit_ranking_quality': benefit_ranking_label(row),
            'explainability': 'Tree SHAP risk and benefit drivers' if row['model'] == 'XGBoost' else 'Standardized coefficient/logit contribution risk and benefit drivers',
            'model_evaluation_takeaway': 'Prioritization candidate if AUC, calibration, observed-gap direction, and uncertainty are acceptable; ROI is evaluated separately in the business value section.',
        }
    )

model_recommendation_summary = pd.DataFrame(recommendation_rows)
model_recommendation_summary_path = output_folder / 'model_recommendation_summary.csv'
model_recommendation_summary.to_csv(model_recommendation_summary_path, index=False)

print('Model recommendation summary:')
display(model_recommendation_summary)
print('Model recommendation summary written to:', model_recommendation_summary_path)





---
## WRITE-UP OUTPUT FILE INDEX
---


In [41]:
writeup_output_files = [
    output_folder / 'data_review_summary.csv',
    output_folder / 'predictor_data_dictionary.csv',
    output_folder / 'numeric_predictor_summary.csv',
    output_folder / 'categorical_predictor_summary.csv',
    output_folder / 'Predictor_Distributions' / 'numeric_predictor_distributions.pdf',
    output_folder / 'Predictor_Distributions' / 'categorical_predictor_distributions.pdf',
    output_folder / 'Predictor_Distributions' / 'predictor_distribution_visual_index.csv',
    output_folder / 'model_evaluation_summary.csv',
    output_folder / 'model_recommendation_summary.csv',
    output_folder / 'factual_event_count_summary.csv',
    output_folder / 'factual_prediction_separation.csv',
    output_folder / 'factual_event_rate_threshold_classification.csv',
    output_folder / 'factual_prediction_ranges.csv',
]

for folder in [xgboost_output_folder, glmnet_output_folder]:
    writeup_output_files.extend(
        [
            folder / 'uplift_decile_summary.csv',
            folder / 'uplift_roi_by_decile.csv',
            folder / 'model_brier_scores.csv',
            folder / 'calibration_summary.csv',
            folder / 'calibration_by_decile.csv',
            folder / 'uplift_observed_gap_by_decile.csv',
            folder / 'uplift_curve_by_decile.csv',
            folder / 'uplift_curve_summary.csv',
            folder / 'top_benefit_decile_summary.csv',
            folder / 'shap_importance_treated_control_models.csv',
            folder / 'shap_importance_benefit_score.csv',
            folder / 'dashboard_calibration_plot.png',
            folder / 'dashboard_observed_gap_by_decile.png',
            folder / 'dashboard_uplift_curve_by_decile.png',
            folder / 'dashboard_shap_benefit_score.png',
        ]
    )

writeup_output_index = pd.DataFrame(
    {
        'output_file': [str(path) for path in writeup_output_files],
        'exists': [path.exists() for path in writeup_output_files],
    }
)

print('Write-up output file index:')
display(writeup_output_index)
print('\n'.join(writeup_output_index['output_file']))

